---
title: 'Ray GPU · 单卡任务与双卡 Actor'
---


# Ray GPU：从资源分配到真实 CUDA 计算

**状态：完整教学 Notebook，实验未运行。** 双卡 RTX6000 Pro、96 核 CPU 来自用户自述，实际设备信息由运行记录确认。
先完成 [CPU Notebook](cpu.ipynb) 的 Task、Actor 和资源单元，再按 **讲解 → 预测 → 代码 → 观察** 阅读本页。

在服务器已有、支持该 GPU 的 CUDA 版 PyTorch 环境中安装主题依赖，并选择对应 kernel。
工作目录为本 Notebook 所在目录。PyTorch/CUDA/驱动版本不由文档预设。

```bash
python -m pip install -r topics/distributed-runtime/ray/requirements.txt
python -m ipykernel install --user --name ai-infra-ray-gpu --display-name "AI Infra · Ray GPU"
nvidia-smi
```

若尚未准备 PyTorch，按 [官方安装说明](https://pytorch.org/get-started/locally/)与实际驱动选择环境。
网站展示保存的输出，服务器 kernel 负责运行。小向量只验证分配与正确性，不比较 GPU 算力。


## 环境：先把观察的来源记下来

在服务器选择安装了本主题依赖的 Python kernel，按顺序运行单元格。修改代码或参数后先保存 Notebook，再开始新的运行记录。
工作目录应为这个 Notebook 所在的 `topics/distributed-runtime/ray/`；VS Code Remote SSH 已配置 `${fileDirname}`。
JupyterLab 从仓库启动时，先把 kernel 工作目录切换到这个目录。
网站只展示 Notebook 与已保存输出，不启动 kernel。

本轮实验未运行。服务器的双卡 RTX6000 Pro、96 核 CPU 来自你的自述；实际资源以下面的记录为准。
默认 `NUM_CPUS=4` 是教学准入容量，等待时长和任务数也都是教学参数。


In [ ]:
import json
import os
import platform
import socket
import subprocess
import sys
import threading
import time
import traceback
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

import ray
from IPython.display import display

TOPIC = Path.cwd().resolve()
assert (TOPIC / "gpu.ipynb").is_file(), "请把 kernel 工作目录切换到 Ray Notebook 所在目录"
REPO = TOPIC.parents[2]
NUM_CPUS = 4
TIMEOUT = 120
owns_runtime = False


### 运行记录也是实验的一部分

每次初始化创建新的 `.runs/` 目录。下面记下实际版本、Git revision、未提交修改和环境；记录不推断机器性能。
`observe()` 给每个运行单元保存成功或异常。失败时关闭本 Notebook 创建的 Ray 运行时，修复后从初始化重新运行。
定义函数和类的单元若有错误，Jupyter 会保留错误输出；保存 Notebook 才能把这些输出取回。


In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
RUN_DIR = REPO / ".runs" / "ray-gpu" / stamp
RUN_DIR.mkdir(parents=True, exist_ok=False)
run = {
    "kind": "Ray gpu Notebook learning experiment",
    "notebook": str(TOPIC / "gpu.ipynb"),
    "status": "running",
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "parameters": {"num_cpus": NUM_CPUS, "timeout_seconds": TIMEOUT},
    "environment": {
        "hostname": socket.gethostname(), "platform": platform.platform(),
        "python": platform.python_version(), "python_executable": sys.executable, "ray": ray.__version__,
        "host_cpu_count": os.cpu_count(),
        "cpu_affinity_count": len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None,
        "git_revision": subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, text=True, capture_output=True, check=True).stdout.strip(),
        "git_status": subprocess.run(["git", "status", "--porcelain"], cwd=REPO, text=True, capture_output=True, check=True).stdout.splitlines(),
    },
    "clock": "time.perf_counter_ns; event times comparable on this single node",
    "steps": [],
}

def save_record():
    (RUN_DIR / "run.json").write_text(json.dumps(run, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

@contextmanager
def observe(name):
    global owns_runtime
    step = {"name": name, "status": "running", "started_ns": time.perf_counter_ns()}
    run["steps"].append(step)
    try:
        yield step
        step["status"] = "passed"
    except BaseException as error:
        step["status"] = "failed"
        step["error"] = {"type": type(error).__name__, "message": str(error), "traceback": traceback.format_exc()}
        run["status"] = "failed"
        if owns_runtime:
            ray.shutdown()
            owns_runtime = False
        raise
    finally:
        step["finished_ns"] = time.perf_counter_ns()
        save_record()

def event(label):
    return {"label": label, "at_ns": time.perf_counter_ns(), "hostname": socket.gethostname(),
            "pid": os.getpid(), "thread_id": threading.get_ident()}

save_record()
display(run["environment"])
print("Results:", RUN_DIR)


### 建立单节点运行时

这里显式使用 `address="local"` 创建本次实验的运行时，CPU 教学使用 128 MiB 对象存储。
`NUM_CPUS` 是逻辑资源数量，不是物理核绑定或线程上限。
如果 kernel 已连接 Ray，请先完成自己的清理或重启 kernel，再开始本次 Notebook。
资料版本：Ray 2.59.0，查阅 2026-10-02。[官方资源模型](https://docs.ray.io/en/latest/ray-core/scheduling/resources.html)


In [ ]:
with observe("runtime_start") as step:
    if ray.is_initialized():
        raise RuntimeError("kernel 已有 Ray 连接；请先清理自己的运行时或重启 kernel")
    ray.init(address="local", num_cpus=NUM_CPUS, include_dashboard=False,
             object_store_memory=128 * 1024**2)
    owns_runtime = True
    run["environment"]["ray_resources"] = ray.cluster_resources()
    step["resources"] = ray.cluster_resources()
    display(step["resources"])


## 1 · 两个层次：Ray 分配资源，PyTorch 执行计算

声明 GPU 资源不会把普通 Python 算术变成 CUDA 运算。
Ray 选择可用设备并设置 worker 的可见性，PyTorch 按设备上的张量运行算子。
先核查实际软件与驱动。设备能被枚举不等于 kernel 一定兼容，后面的数值计算才是进一步验证。
[GPU assignment](https://docs.ray.io/en/latest/ray-core/scheduling/accelerators.html)

**预测：** Notebook kernel 能看到的设备数，是否必须与每个 Ray worker 相同？


In [ ]:
import torch

with observe("gpu_environment") as step:
    driver_info = {
        "torch": torch.__version__, "torch_cuda_build": torch.version.cuda,
        "driver_cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "driver_cuda_available": torch.cuda.is_available(),
        "driver_cuda_device_count": torch.cuda.device_count(),
        "nvidia_smi": subprocess.run(
            ["nvidia-smi", "--query-gpu=index,name,uuid,memory.total,driver_version", "--format=csv"],
            text=True, capture_output=True, check=True).stdout.splitlines(),
    }
    run["environment"].update(driver_info)
    step["environment"] = driver_info
    resources = ray.cluster_resources()
    if "GPU" not in resources or resources["GPU"] < 1:
        raise RuntimeError(f"至少需要一张 Ray 可见 GPU；实际资源为 {resources}")
    display(driver_info)


## 2 · 单 GPU Task：把设备声明和张量操作连接起来

下面的普通函数先核查 worker 的设备视图。每个工作声明一张 GPU，所以检查一个 Ray 分配 ID、一个 PyTorch 可见设备。
Ray ID、`CUDA_VISIBLE_DEVICES`、进程内编号与主机物理编号需要分别理解；已有设备掩码时尤其不能直接等同。
[逻辑资源与可见性](https://docs.ray.io/en/latest/ray-core/scheduling/resources.html)


In [ ]:
def device_info():
    assigned = ray.get_runtime_context().get_accelerator_ids()["GPU"]
    assert len(assigned) == 1, assigned
    assert torch.cuda.is_available(), "该 worker 中 PyTorch 无法使用分配的 GPU"
    assert torch.cuda.device_count() == 1, "该 worker 应只看到一张分配的 GPU"
    properties = torch.cuda.get_device_properties(0)
    return {"ray_gpu_ids": assigned, "cuda_visible_devices": os.environ["CUDA_VISIBLE_DEVICES"],
            "local_device": "cuda:0", "name": properties.name,
            "total_memory_bytes": properties.total_memory,
            "compute_capability": list(torch.cuda.get_device_capability(0)), "pid": os.getpid()}


### 明确写出计算与同步

这里用 4096 个 FP32 元素做 `arange + ones`。整数范围内的期望值可以精确比较，检查和性能测量是不同目标。
CUDA 提交是异步的；完成事件在 `torch.cuda.synchronize()` 后记录，再把结果拿到 CPU 核对。
这些主机事件包含同步成本，预先提交的缓冲区初始化也可能在同步时被等待；它们不是 CUDA Events 的纯 kernel 时间。

**预测：** 声明 `num_gpus=1` 后，计算仍需要哪些 PyTorch 语句才能发生在 GPU？


In [ ]:
ELEMENTS = 4096
run["parameters"].update(elements=ELEMENTS, dtype="float32")

def checked_add(x, y):
    started = event("host_before_add")
    result = x + y
    torch.cuda.synchronize()
    synchronized = event("device_synchronized")
    expected = torch.arange(ELEMENTS, dtype=torch.float32) + 1
    torch.testing.assert_close(result.cpu(), expected, rtol=0, atol=0)
    return {"correct": True, "events": [started, synchronized, event("cpu_validation_end")]}

@ray.remote(num_cpus=1, num_gpus=1)
def vector_add():
    device = device_info()
    x = torch.arange(ELEMENTS, dtype=torch.float32, device="cuda:0")
    y = torch.ones_like(x)
    return {"device": device, "computation": checked_add(x, y)}


现在提交一次 Task。driver 等的是引用指向的结果；tensor 运算在 worker 内进行。
先预测返回信息里的 PID、Ray GPU ID 与 `cuda:0`，再运行。


In [ ]:
with observe("one_gpu_task") as step:
    gpu_ref = vector_add.remote()
    task_result = ray.get(gpu_ref, timeout=TIMEOUT)
    assert task_result["computation"]["correct"]
    step["result"] = task_result
    display(task_result)


**观察与复述：** 区分资源声明、设备选择、真正的加法、同步和 CPU 核对。为什么 worker 的 `cuda:0` 不一定是主机的物理 0 号卡？

## 3 · 双 GPU Actor：每个进程保留自己的设备与缓冲区

Actor 创建后保留实例和已分配资源。两个 Actor 各声明一张 GPU，持续持有各自设备；这比两个瞬间完成的 Task 更适合观察双卡分配。
只申请两张 GPU 不会自动形成 tensor parallel；这里只验证两份独立工作。[Actor](https://docs.ray.io/en/latest/ray-core/actors.html)

```mermaid
---
config:
  theme: base
  themeVariables:
    background: '#FAF8F4'
    primaryColor: '#E4F0EB'
    primaryTextColor: '#292524'
    primaryBorderColor: '#0F766E'
    lineColor: '#6B635A'
    fontFamily: 'system-ui, sans-serif'
---
flowchart LR
    driver[Driver]:::communication -->|create| actorA[Actor A process]:::compute
    driver -->|create| actorB[Actor B process]:::compute
    actorA --> localA[Buffers on cuda:0]:::memory --> gpuA[Assigned GPU A]:::compute
    actorB --> localB[Buffers on cuda:0]:::memory --> gpuB[Assigned GPU B]:::compute
    classDef compute fill:#EAF0F8,stroke:#315C8C,color:#292524
    classDef memory fill:#F8EFDA,stroke:#9A6700,color:#292524
    classDef communication fill:#E4F0EB,stroke:#0F766E,color:#292524
```

图为教学示意，假设两个 Actor 被分配不同 GPU。每个进程的本地可见设备都从 `cuda:0` 开始；实际映射以下面的记录为准。

**预测：** 两个进程都写 `cuda:0`，是否就会用到同一张卡？


In [ ]:
@ray.remote(num_cpus=1, num_gpus=1)
class VectorActor:
    """在一张分配的 GPU 上保留输入缓冲区和调用计数。"""
    def __init__(self):
        self.device = device_info()
        self.x = torch.arange(ELEMENTS, dtype=torch.float32, device="cuda:0")
        self.y = torch.ones_like(self.x)
        self.calls = 0

    def step(self):
        computation = checked_add(self.x, self.y)
        self.calls += 1
        return {"device": self.device, "calls": self.calls, "computation": computation}


### 先检查资源，再提交两个 Actor

默认 4 个逻辑 CPU 足够两个各声明 1 CPU 的 Actor。下面在提交前检查至少 2 张可见 GPU 与 2 个逻辑 CPU，避免资源不足时一直等待。
Actor 实例保留到第二轮，才能直接观察状态与资源继续持有。


In [ ]:
with observe("two_gpu_actors_first_round") as step:
    resources = ray.cluster_resources()
    if "GPU" not in resources or resources["GPU"] < 2 or resources["CPU"] < 2:
        raise RuntimeError(f"双 Actor 需要至少 2 GPU 与 2 CPU；实际资源为 {resources}")
    actors = [VectorActor.remote() for _ in range(2)]
    first_round = ray.get([actor.step.remote() for actor in actors], timeout=TIMEOUT)
    assigned = [set(result["device"]["ray_gpu_ids"]) for result in first_round]
    assert assigned[0].isdisjoint(assigned[1]), "两个 Actor 应持有不同的 Ray GPU 分配"
    assert len({result["device"]["pid"] for result in first_round}) == 2
    assert [result["calls"] for result in first_round] == [1, 1]
    step["results"] = first_round
    display(first_round)


### 第二轮：检查状态与映射是否延续

每个 Actor 的缓冲区只在 `__init__` 中分配。再次调用方法，实例计数继续增长。
检查同一 Actor 的 PID 与设备映射不变，不预设 A 必须是 0 号卡、B 必须是 1 号卡。

**预测：** 第二轮的计数应该怎样？PID 和本地 CUDA 编号分别说明什么？


In [ ]:
with observe("two_gpu_actors_second_round") as step:
    second_round = ray.get([actor.step.remote() for actor in actors], timeout=TIMEOUT)
    for first, second in zip(first_round, second_round):
        assert second["calls"] == 2
        assert first["device"] == second["device"]
        assert second["computation"]["correct"]
    step["results"] = second_round
    display(second_round)


### 释放实例，归还本运行时的逻辑资源

先从两轮记录解释“持久状态”和“两个进程的设备视图”，再清理 Actor。
GPU 资源声明不是显存配额，也不是与其他程序之间的物理设备独占承诺。


In [ ]:
with observe("release_gpu_actors") as step:
    for actor in actors:
        ray.kill(actor)
    step["released_actor_count"] = len(actors)


## 4 · 接回真实 AI 系统

Ray Core 编排进程、方法与依赖；PyTorch/CUDA 执行张量操作；NCCL 承担张量 collective。
vLLM 的请求、KV 和生成调度，以及 verl 的训练/rollout/评分角色，属于需要进一步追踪的系统行为。
两张卡上各算一份向量并不等于模型并行；后续可连接 [通信与分布式训练](../../distributed-training/README.md)与 [verl 材料](../../../paths/ai-infra/14-post-training-rlhf.md#ai-14-7)。

本实验只验证分配、状态和正确性。小向量的计时不支持算力排序；性能研究需要另设稳定负载、预热、重复测量和完整环境记录。


## 保存、清理与复述

先回答本页的观察问题，写下自己的解释和遗漏前提，再执行清理单元。步骤的 `passed` 表示该运行单元完成；会话的 `finished` 表示已保存并清理，不代表所有单元已执行或已经掌握。
重新开始一个完整实验时，重启 kernel 并从头运行，使用新的运行目录。


In [ ]:
if owns_runtime:
    ray.shutdown()
    owns_runtime = False
run["status"] = "finished" if all(step["status"] == "passed" for step in run["steps"]) else "failed"
run["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
save_record()
display({"status": run["status"], "record": str(RUN_DIR / "run.json")})


### 回收学习成果

保存 Notebook 的真实输出；把本次 `.runs/` 目录和 Notebook 一起取回本机。服务器修改了代码或讲解时，先比较差异，再合并。
例如 SSH 别名为 `gpu-box`、服务器仓库为 `~/ai-infra-learning`，在本机仓库根目录执行：

```bash
mkdir -p .runs/from-ray-server
rsync -av gpu-box:~/ai-infra-learning/topics/distributed-runtime/ray/gpu.ipynb .runs/from-ray-server/
rsync -av gpu-box:~/ai-infra-learning/.runs/ray-gpu/ .runs/from-ray-server/ray-gpu/
```

精选记录放主题 `results/`；图保留 SVG 与绘图单元，Notebook 保存 PNG 输出。你的复述填在下面，未解决的问题进入 [GAPS](../../../GAPS.md)。

- **我的原先解释：** 待填写。
- **实际观察与记录位置：** 未运行，待填写。
- **修正后的解释与适用前提：** 待填写。
- **下一步问题：** 待填写。


## 来源

Ray [Resources](https://docs.ray.io/en/latest/ray-core/scheduling/resources.html)、[Accelerators](https://docs.ray.io/en/latest/ray-core/scheduling/accelerators.html)和 [Actors](https://docs.ray.io/en/latest/ray-core/actors.html)，按 2.59.0 核查，日期 2026-10-02。
GPU 实际 PyTorch、CUDA build、驱动与设备型号以本次运行记录为准。
